In [68]:
import re
import json
from collections import Counter

print("Prompt Engineering Lab is ready!")


Prompt Engineering Lab is ready!


Introduction to prompt engineering

In [69]:

# STEP 2: Create a structured prompt

def create_prompt(role, task, context, output_format):
    prompt = f"""
Role:
{role}

Task:
{task}

Context:
{context}

Required output format:
{output_format}
"""
    return prompt.strip()


prompt = create_prompt(
    role="You are an NLP engineer.",
    task="Classify the sentiment of the supplied review.",
    context="Review: The application is useful and easy to use.",
    output_format="Return JSON with sentiment and explanation."
)

print(prompt)


Role:
You are an NLP engineer.

Task:
Classify the sentiment of the supplied review.

Context:
Review: The application is useful and easy to use.

Required output format:
Return JSON with sentiment and explanation.


Zero-shot prompting

In [70]:

# STEP 3: Zero-shot prompting

def zero_shot_prompt(text, task="Classify the sentiment"):
    return f"""
Task: {task}

Text: {text}

Choose one label: Positive, Negative, or Neutral.
Return only the label.
""".strip()


review = "The customer support was excellent."
prompt = zero_shot_prompt(review)

print(prompt)


Task: Classify the sentiment

Text: The customer support was excellent.

Choose one label: Positive, Negative, or Neutral.
Return only the label.


Few-shot prompting

In [71]:

# STEP 4: Few-shot prompting

def few_shot_prompt(text):
    return f"""
Classify each review as Positive or Negative.

Example 1:
Review: "The product is amazing."
Label: Positive

Example 2:
Review: "The service was terrible."
Label: Negative

Example 3:
Review: "I love the quality."
Label: Positive

Now classify this review:
Review: "{text}"
Label:
""".strip()


print(few_shot_prompt("The delivery was slow and disappointing."))


Classify each review as Positive or Negative.

Example 1:
Review: "The product is amazing."
Label: Positive

Example 2:
Review: "The service was terrible."
Label: Negative

Example 3:
Review: "I love the quality."
Label: Positive

Now classify this review:
Review: "The delivery was slow and disappointing."
Label:


Chain-of-Thought (CoT) prompting

In [72]:

# STEP 5: Structured reasoning prompt

def cot_prompt(question):
    return f"""
Solve the following problem.

Question:
{question}

Instructions:
1. Identify the relevant information.
2. Show a concise, verifiable calculation.
3. Check the result.
4. State the final answer clearly.
""".strip()


question = (
    "A shop sells 3 notebooks for Rs. 90 each. "
    "What is the total cost?"
)

print(cot_prompt(question))


Solve the following problem.

Question:
A shop sells 3 notebooks for Rs. 90 each. What is the total cost?

Instructions:
1. Identify the relevant information.
2. Show a concise, verifiable calculation.
3. Check the result.
4. State the final answer clearly.


Self-consistency prompting

In [73]:

# STEP 6: Self-consistency voting

def majority_vote(answers):
    counts = Counter(answers)
    winner, votes = counts.most_common(1)[0]
    return winner, votes, counts


candidate_answers = [
    "270",
    "270",
    "260",
    "270",
    "270"
]

winner, votes, counts = majority_vote(candidate_answers)

print("All answers:", candidate_answers)
print("Vote counts:", dict(counts))
print("Selected answer:", winner)
print("Votes for selected answer:", votes)


All answers: ['270', '270', '260', '270', '270']
Vote counts: {'270': 4, '260': 1}
Selected answer: 270
Votes for selected answer: 4


In [74]:

# STEP 6.1: Generate a self-consistency prompt

def self_consistency_prompt(question, samples=5):
    return f"""
Question:
{question}

Generate {samples} independent candidate solutions.
For each candidate, provide a concise calculation and a final answer.
Use the same units and final-answer format for every candidate.
""".strip()


print(self_consistency_prompt(
    "A shop sells 3 notebooks for Rs. 90 each. Find the total cost."
))


Question:
A shop sells 3 notebooks for Rs. 90 each. Find the total cost.

Generate 5 independent candidate solutions.
For each candidate, provide a concise calculation and a final answer.
Use the same units and final-answer format for every candidate.


Zero-Shot Chain-of-Thought prompting

In [75]:

# STEP 7: Zero-shot CoT

def zero_shot_cot_prompt(question):
    return f"""
Question:
{question}

Solve the problem carefully.
Show a concise, checkable solution and verify the final answer.
End with: Final answer: ...
""".strip()


print(zero_shot_cot_prompt(
    "A bus travels 60 km in one hour. "
    "How far will it travel in 3.5 hours at the same speed?"
))


Question:
A bus travels 60 km in one hour. How far will it travel in 3.5 hours at the same speed?

Solve the problem carefully.
Show a concise, checkable solution and verify the final answer.
End with: Final answer: ...


ReAct (Reasoning + Acting) prompting

In [76]:

# STEP 8: ReAct-style tool execution

def calculator(expression):
    # Only permit simple arithmetic characters.
    if not re.fullmatch(r"[0-9+\-*/(). ]+", expression):
        raise ValueError("Unsupported calculator expression")

    return eval(expression, {"__builtins__": {}}, {})


def react_demo(question, expression):
    print("Question:", question)

    # Action
    print("\nAction: calculator")
    print("Input:", expression)

    # Observation
    observation = calculator(expression)
    print("Observation:", observation)

    # Final response
    print("\nFinal answer:", observation)


react_demo(
    "What is the total cost of 3 notebooks at Rs. 90 each?",
    "3 * 90"
)


Question: What is the total cost of 3 notebooks at Rs. 90 each?

Action: calculator
Input: 3 * 90
Observation: 270

Final answer: 270


Prompt chaining

In [77]:

# STEP 9: Prompt chaining

def stage_1_extract_topic(review):
    return f"""
Extract the main topic from this review.
Review: {review}
Return a short topic phrase.
""".strip()


def stage_2_classify_sentiment(topic, review):
    return f"""
Classify the sentiment of the review as Positive, Negative, or Neutral.
Topic: {topic}
Review: {review}
Return the label and a short justification.
""".strip()


def stage_3_create_report(topic, sentiment_result):
    return f"""
Create a concise customer-feedback report.

Topic: {topic}
Sentiment analysis: {sentiment_result}

Include the topic, sentiment, and one recommended action.
""".strip()


review = "The checkout process is confusing, and payment keeps failing."

# Each stage creates the prompt for the next stage.
topic_prompt = stage_1_extract_topic(review)
print("STAGE 1 PROMPT:\n", topic_prompt)

# Demonstration placeholder for the output of a language model:
topic = "Checkout and payment"

sentiment_prompt = stage_2_classify_sentiment(topic, review)
print("\nSTAGE 2 PROMPT:\n", sentiment_prompt)

# Demonstration placeholder for the model's output:
sentiment_result = "Negative — users report confusing checkout and failed payments."

report_prompt = stage_3_create_report(topic, sentiment_result)
print("\nSTAGE 3 PROMPT:\n", report_prompt)


STAGE 1 PROMPT:
 Extract the main topic from this review.
Review: The checkout process is confusing, and payment keeps failing.
Return a short topic phrase.

STAGE 2 PROMPT:
 Classify the sentiment of the review as Positive, Negative, or Neutral.
Topic: Checkout and payment
Review: The checkout process is confusing, and payment keeps failing.
Return the label and a short justification.

STAGE 3 PROMPT:
 Create a concise customer-feedback report.

Topic: Checkout and payment
Sentiment analysis: Negative — users report confusing checkout and failed payments.

Include the topic, sentiment, and one recommended action.


Role-based prompting

In [78]:

# STEP 10: Role-based prompting

def role_based_prompt(role, audience, task, constraints):
    return f"""
Role:
{role}

Target audience:
{audience}

Task:
{task}

Constraints:
{constraints}
""".strip()


prompt = role_based_prompt(
    role="Senior NLP Engineer",
    audience="Beginner Python developers",
    task="Explain word embeddings and give a simple example.",
    constraints=(
        "Use clear language, include Python code, "
        "and distinguish embeddings from one-hot encoding."
    )
)

print(prompt)


Role:
Senior NLP Engineer

Target audience:
Beginner Python developers

Task:
Explain word embeddings and give a simple example.

Constraints:
Use clear language, include Python code, and distinguish embeddings from one-hot encoding.


Tree of Thought (ToT) prompting

In [79]:

# STEP 11: Tree of Thought candidate evaluation

problem = "Choose a beginner-friendly NLP portfolio project."

candidates = [
    {
        "name": "Sentiment Analysis",
        "difficulty": 2,
        "portfolio_value": 3,
        "implementation_cost": 1
    },
    {
        "name": "Semantic Search Engine",
        "difficulty": 4,
        "portfolio_value": 5,
        "implementation_cost": 3
    },
    {
        "name": "AI Code Review Assistant",
        "difficulty": 5,
        "portfolio_value": 5,
        "implementation_cost": 4
    }
]

def score_candidate(candidate):
    # Higher score is better.
    # Reward portfolio value; penalize difficulty and cost.
    return (
        3 * candidate["portfolio_value"]
        - candidate["difficulty"]
        - candidate["implementation_cost"]
    )


ranked = sorted(
    candidates,
    key=score_candidate,
    reverse=True
)

print("Problem:", problem)
print("\nCandidate evaluation:")

for candidate in ranked:
    print(
        candidate["name"],
        "| Score:", score_candidate(candidate)
    )

print("\nSelected approach:", ranked[0]["name"])


Problem: Choose a beginner-friendly NLP portfolio project.

Candidate evaluation:
Semantic Search Engine | Score: 8
Sentiment Analysis | Score: 6
AI Code Review Assistant | Score: 6

Selected approach: Semantic Search Engine


Test all prompt templates together

In [80]:

# STEP 12: Prompt engineering mini-lab

question = "Explain how a neural network learns."

prompts = {
    "Zero-Shot": zero_shot_prompt(question, "Explain the concept"),
    "Few-Shot": few_shot_prompt(question),
    "Chain-of-Thought": cot_prompt(question),
    "Self-Consistency": self_consistency_prompt(question, samples=3),
    "Zero-Shot CoT": zero_shot_cot_prompt(question),
    "ReAct": (
        "Answer the question using tools when necessary: "
        + question
    ),
    "Prompt Chaining": stage_1_extract_topic(question),
    "Role-Based": role_based_prompt(
        "AI/ML Engineer",
        "Beginner",
        question,
        "Use a practical example."
    ),
    "Tree of Thought": (
        "Generate multiple possible explanations, evaluate them, "
        "and select the clearest one: " + question
    )
}

for technique, prompt in prompts.items():
    print("=" * 60)
    print(technique.upper())
    print("=" * 60)
    print(prompt[:500])
    print()


ZERO-SHOT
Task: Explain the concept

Text: Explain how a neural network learns.

Choose one label: Positive, Negative, or Neutral.
Return only the label.

FEW-SHOT
Classify each review as Positive or Negative.

Example 1:
Review: "The product is amazing."
Label: Positive

Example 2:
Review: "The service was terrible."
Label: Negative

Example 3:
Review: "I love the quality."
Label: Positive

Now classify this review:
Review: "Explain how a neural network learns."
Label:

CHAIN-OF-THOUGHT
Solve the following problem.

Question:
Explain how a neural network learns.

Instructions:
1. Identify the relevant information.
2. Show a concise, verifiable calculation.
3. Check the result.
4. State the final answer clearly.

SELF-CONSISTENCY
Question:
Explain how a neural network learns.

Generate 3 independent candidate solutions.
For each candidate, provide a concise calculation and a final answer.
Use the same units and final-answer format for every candidate.

ZERO-SHOT COT
Question:
Explain h